In [0]:
dbutils.widgets.text(
    "process_date",
    "2026-10-08"
)

In [0]:
dbutils.widgets.text("environment","dev")

In [0]:
environment=dbutils.widgets.get("environment")

In [0]:
process_date = dbutils.widgets.get(
    "process_date"
)

print("Process Date:", process_date)

Process Date: 2026-10-08


In [0]:
from datetime import datetime

process_date_obj = datetime.strptime(
    process_date,
    "%Y-%m-%d"
).date()



Process Date: 2026-10-08
Year: 2026
Month: 10
Day: 8


In [0]:
print("Process Date:", process_date_obj)
print("Year:", process_date_obj.year)
print("Month:", process_date_obj.month)
print("Day:", process_date_obj.day)

Process Date: 2026-10-08
Year: 2026
Month: 10
Day: 8


In [0]:
config = {
    "environment": environment,
    "process_date": process_date
}

print("Pipeline Configuration")
print("=======================")

for key, value in config.items():
    print(f"{key}: {value}")

Pipeline Configuration
environment: dev
process_date: 2026-10-08


In [0]:

storage_account = "nestlesalesdatalake"
container = "customer-sales-datalake"

base_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/"
)

landing_base = base_path + "landing/"
manifest_path=base_path+"/metadata/file_manifest/"

# for item in dbutils.fs.ls(landing_base):
#     print(item.name)
#     print(item.path)
#     print(item.isDir())

def discover(path):
    discovered=[]
    for item in dbutils.fs.ls(path):
        if item.isDir():
            discovered.extend(discover(item.path))
        elif item.path.endswith("json"):
            discovered.append(item.path)
    return discovered
res=discover(landing_base)
for x in res:
    print(x)

abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=08/orders_20261008.json
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json


In [0]:
dataset=["orders","customers","products"]
discovered_files=[]
for x in dataset:
    path=landing_base + f'{x}'
    y=discover(path)
    for z in y:
        discovered_files.append()





discovered_files = []

for dataset in datasets:

    dataset_path = f"{landing_base}{dataset}/"

    try:
        dataset_files = discover_files(dataset_path)

        for file_path in dataset_files:
            discovered_files.append({
                "dataset": dataset,
                "file_name": file_path.rstrip("/").split("/")[-1],
                "file_path": file_path
            })

        print(
            f"[OK] {dataset}: "
            f"{len(dataset_files)} JSON files discovered"
        )

    except Exception as e:
        raise RuntimeError(
            f"Failed to discover files under {dataset_path}"
        ) from e



    


[['abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json', 'abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=08/orders_20261008.json'], ['abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json', 'abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json'], ['abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json', 'abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json']]


In [0]:
# 5. Load the existing manifest

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    IntegerType,
    DateType
)
from delta import DeltaTable

In [0]:
# Define the manifest schema
manifest_schema = StructType([
    StructField("dataset", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("first_seen", TimestampType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("retry_count", IntegerType(), True),
    StructField("error_message", StringType(), True),
    StructField("data_date", DateType(), True)
])

In [0]:
# Check whether the manifest Delta table exists
if DeltaTable.isDeltaTable(spark, manifest_path):

    manifest_df = (
        spark.read
        .format("delta")
        .load(manifest_path)
    )

    print("Existing manifest loaded successfully.")

else:

    manifest_df = spark.createDataFrame(
        [],
        schema=manifest_schema
    )

    print("Manifest does not exist. Created an empty DataFrame.")

# Display the current manifest
print("Manifest records:", manifest_df.count())

display(
    manifest_df.orderBy("dataset", "file_path")
)


Existing manifest loaded successfully.
Manifest records: 6


dataset,file_name,file_path,batch_id,status,first_seen,processed_at,run_id,retry_count,error_message,data_date
customers,customers_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json,CUSTOMERS_20261007,SUCCESS,2026-10-08T07:34:41.899Z,2026-10-08T17:39:02.063Z,RUN_20261008133237_60df54dc,0,null,2026-10-07
customers,customers_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json,CUSTOMERS_20261008,PROCESSING,2026-10-08T17:31:02.535Z,null,FILE_MANIFEST_20261008173013_69ab4915,0,null,2026-10-08
orders,orders_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json,ORDERS_20261007,SUCCESS,2026-10-08T07:34:41.899Z,2026-10-08T11:41:51.399Z,RUN_20261008112412_7f871f96,0,null,2026-10-07
orders,orders_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=08/orders_20261008.json,ORDERS_20261008,SUCCESS,2026-10-08T17:31:02.535Z,2026-10-08T17:32:02.683Z,RUN_20261008173146_b718226c,0,null,2026-10-08
products,products_20261007.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json,PRODUCTS_20261007,SUCCESS,2026-10-08T07:34:41.899Z,2026-10-08T13:43:16.460Z,RUN_20261008134304_d43a32ad,0,null,2026-10-07
products,products_20261008.json,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json,PRODUCTS_20261008,SUCCESS,2026-10-08T17:31:02.535Z,2026-10-08T17:32:07.686Z,RUN_20261008173155_c8f94f0f,0,null,2026-10-08


In [0]:
from pyspark.sql import functions as F

# Select the file identity columns from the existing manifest
existing_files_df = (
    manifest_df
    .select("dataset", "file_path")
    .dropDuplicates()
)
display(existing_files_df)

dataset,file_path
orders,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=07/orders_20261007.json
products,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=07/products_20261007.json
customers,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=08/customers_20261008.json
orders,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=08/orders_20261008.json
products,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/year=2026/month=10/day=08/products_20261008.json
customers,abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/year=2026/month=10/day=07/customers_20261007.json


In [0]:



# Find discovered files that do not exist in the manifest
new_files_df = (
    discovered_files_df.alias("discovered")
    .join(
        existing_files_df.alias("existing"),
        on=["dataset", "file_path"],
        how="left_anti"
    )
)

# Display the new files
new_files_count = new_files_df.count()

print("Total discovered files:", discovered_files_df.count())
print("Files already registered:", existing_files_df.count())
print("New files to register:", new_files_count)

display(
    new_files_df.orderBy("dataset", "file_path")
)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5405481531646596>, line 6
      4 # Find discovered files that do not exist in the manifest
      5 new_files_df = (
----> 6     discovered_files_df.alias("discovered")
      7     .join(
      8         existing_files_df.alias("existing"),
      9         on=["dataset", "file_path"],
     10         how="left_anti"
     11     )
     12 )
     14 # Display the new files
     15 new_files_count = new_files_df.count()

NameError: name 'discovered_files_df' is not defined